# 3주차 실습 (1) - Information Theory from scratch

순수 Python으로 정보이론의 핵심 양들을 직접 구현하고 관계를 확인한다.

- 엔트로피 `H(P)`, 크로스 엔트로피 `H(P,Q)`, KL 발산 `D_KL(P||Q)`
- 핵심 항등식: **`H(P,Q) = H(P) + D_KL(P||Q)`**
- 크로스 엔트로피 = 음의 로그우도(NLL) 임을 수치로 확인

## Step 1. 정보량과 엔트로피

- 정보량(놀람): `I(x) = -log p(x)` — 드문 사건일수록 크고, 확실한 사건(p=1)은 0
- 엔트로피: `H(P) = -Σ p log p` — 모든 결과에 대한 정보량의 기댓값
- log 밑이 2면 bit, e면 nat

In [ ]:
import math


def information_content(p, base=2):
    """사건 확률 p의 정보량(놀람) -log_base(p)."""
    if p <= 0 or p > 1:
        return float("inf") if p <= 0 else 0.0
    return -math.log(p) / math.log(base)


def entropy(probs, base=2):
    """분포의 평균 놀람 H(P) = -sum p*log p."""
    return sum(p * information_content(p, base) for p in probs if p > 0)


print(f"공정한 동전:  {entropy([0.5, 0.5]):.4f} bits")   # 1.0
print(f"편향 동전:    {entropy([0.99, 0.01]):.4f} bits")  # 0.08
print(f"공정 주사위:  {entropy([1 / 6] * 6):.4f} bits")   # 2.585

## Step 2. 크로스 엔트로피와 KL 발산

- `H(P, Q) = -Σ p log q` : 진짜 분포 P를 모델 분포 Q로 인코딩할 때의 평균 놀람
- `D_KL(P||Q) = H(P,Q) - H(P)` : Q를 써서 낭비한 여분
- 학습 중 `H(P)`는 상수 → **CE 최소화 = KL 최소화**

In [ ]:
def cross_entropy(p, q, base=2):
    total = 0.0
    for pi, qi in zip(p, q):
        if pi > 0:
            if qi <= 0:
                return float("inf")
            total += pi * (-math.log(qi) / math.log(base))
    return total


def kl_divergence(p, q, base=2):
    return cross_entropy(p, q, base) - entropy(p, base)


p = [0.7, 0.2, 0.1]
for name, q in [("good", [0.6, 0.25, 0.15]), ("bad", [0.1, 0.1, 0.8])]:
    ce, kl, h = cross_entropy(p, q), kl_divergence(p, q), entropy(p)
    print(f"[{name}] CE={ce:.4f}  H={h:.4f}  KL={kl:.4f}  "
          f"H+KL={h + kl:.4f}  일치={math.isclose(ce, h + kl)}")

## Step 3. 분류 손실로서의 크로스 엔트로피 + 퍼플렉서티

- 원-핫 타깃이면 `H(P,Q) = -log q(정답클래스)` 로 단순화 → 이게 곧 CrossEntropyLoss
- 퍼플렉서티 `PP = e^{H(P,Q)}` : "평균적으로 몇 개 선택지 사이에서 헷갈리는가"

In [ ]:
def softmax(logits):
    m = max(logits)
    exps = [math.exp(z - m) for z in logits]
    s = sum(exps)
    return [e / s for e in exps]


def cross_entropy_loss(true_class, logits):
    """원-핫 타깃 CE = -log q(정답). base=e (nat)."""
    return -math.log(softmax(logits)[true_class])


logits, true_class = [2.0, 1.0, 0.1], 0
loss = cross_entropy_loss(true_class, logits)
print(f"softmax     = {[round(x, 4) for x in softmax(logits)]}")
print(f"loss        = {loss:.4f} nats")
print(f"perplexity  = {math.exp(loss):.2f}")

## Step 4. 크로스 엔트로피 == 음의 로그우도(NLL)

1000개 샘플에서 두 값이 (부동소수 오차 범위 내에서) 같음을 확인한다.

In [ ]:
import random

random.seed(42)
n, k = 1000, 3
labels = [random.randint(0, k - 1) for _ in range(n)]
model_logits = [[random.gauss(0, 1) for _ in range(k)] for _ in range(n)]

ce = sum(cross_entropy_loss(y, z) for y, z in zip(labels, model_logits)) / n
nll = -sum(math.log(softmax(z)[y]) for y, z in zip(labels, model_logits)) / n
print(f"Cross-entropy : {ce:.6f}")
print(f"NLL           : {nll:.6f}")
print(f"차이          : {abs(ce - nll):.2e}")

## Step 5. 상호정보량

`I(X;Y) = Σ p(x,y) log( p(x,y) / (p(x)p(y)) )` — 독립이면 0, 의존하면 > 0

In [ ]:
def mutual_information(joint, base=2):
    rows, cols = len(joint), len(joint[0])
    px = [sum(joint[i][j] for j in range(cols)) for i in range(rows)]
    py = [sum(joint[i][j] for i in range(rows)) for j in range(cols)]
    mi = 0.0
    for i in range(rows):
        for j in range(cols):
            pxy = joint[i][j]
            if pxy > 0:
                mi += pxy * math.log(pxy / (px[i] * py[j])) / math.log(base)
    return mi


print(f"독립: {mutual_information([[0.25, 0.25], [0.25, 0.25]]):.4f} bits")  # 0.0
print(f"의존: {mutual_information([[0.45, 0.05], [0.05, 0.45]]):.4f} bits")  # > 0

## 정리

- `torch.nn.CrossEntropyLoss()`가 내부에서 하는 일을 직접 만들어봤다 (softmax + NLL).
- 학습 중 손실이 내려간다 = 모델 분포가 진짜 분포에 가까워지며 낭비하는 nat이 줄어든다.
- 관련 개념 노트: 엔트로피 / 크로스 엔트로피와 KL / 상호정보량 (Obsidian LLM_Wiki)